In [58]:
import json
import re
import csv
import numpy as np

output = []
for i in range(10):
    output.extend(json.load(open(f"../data/responce{i+1}{i+1}.json", "r")))
output.extend(json.load(open(f"../data/responce111.json", "r")))
output.extend(json.load(open(f"../data/responce666.json", "r")))
output.extend(json.load(open(f"../data/responce777.json", "r")))
data = json.load(open("../data/data1.json", "r"))
print(len(output),len(data))

6000 6000


In [59]:
def mistakes(answer_key_id):
    correct_list = "000101101001000"
    True_mistakes = 0
    for i in range(len(correct_list)):
        if int(correct_list[i])!=int(answer_key_id[i]):
            True_mistakes += 1
    return True_mistakes

prompt_types = ["very_pessimistic","pessimistic","neutral","confident","very_confident"]
count = 0

with open("ai_grading100.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["answer_key_id", "true_mistakes", "prompt_type", "ai_estimated_mistakes"])

for i in data:
    answer_key_id = i[:16]
    True_error = mistakes(answer_key_id)
    prompt_type = prompt_types[count%5]
    try:
        ai_correct = int(re.findall(r"frac{\d}\{15\}",output[count])[0][5])
        ai_error = 15-ai_correct
    except IndexError:
        ai_error = "not found"
    with open("ai_grading100.csv", "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([answer_key_id,True_error,prompt_type,ai_error])
    count += 1


In [60]:
with open("ai_grading100.csv", "r") as f:
    ai_grading_list = list(csv.reader(f))
print(ai_grading_list[1:10], len(ai_grading_list))

[['111010010110111 ', '15', 'very_pessimistic', 'not found'], ['111010010110111 ', '15', 'pessimistic', 'not found'], ['111010010110111 ', '15', 'neutral', 'not found'], ['111010010110111 ', '15', 'confident', 'not found'], ['111010010110111 ', '15', 'very_confident', 'not found'], ['111010010110111 ', '15', 'very_pessimistic', 'not found'], ['111010010110111 ', '15', 'pessimistic', 'not found'], ['111010010110111 ', '15', 'neutral', 'not found'], ['111010010110111 ', '15', 'confident', 'not found']] 6001


In [61]:
with open("ai_grading100.csv", "r") as f:
    ai_grading_list = list(csv.reader(f))
print(ai_grading_list[1:10],len(ai_grading_list))
for j in range(7): 
    if j!=0:
        with open(f"ai_grading1{j-1}.csv", "r") as f:
            ai_grading_list = list(csv.reader(f))

    with open(f"ai_grading1{j}.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["answer_key_id", "true_mistakes", "prompt_type", "ai_estimated_mistakes"])

    for i in range(len(output)):
        datapoint = ai_grading_list[i + 1].copy()

        if datapoint[3] == "not found":
            patterns = [
                r"umber of correct answers:\s*(\d+)", 
                r"frac\{(\d+)\}\{15\}", 
                r"(\d+) out of 15",
                r"^(\d+)\s*\n\s*\d+\s*\n\s*[\d.]+",  
                r"umber of incorrect answers: (\d+)",
                "(\d+)\u202f%",
                r"ercentage correct: (\d+\.?\d*)"
                
            ]
            try:
                if j == 3:
                    match = re.match(patterns[j], output[i])
                else:
                    match = re.search(patterns[j], output[i])
                if match:
                    #number of correct
                    if j in (0,1,2):
                        ai_correct = int(match.group(1))
                        datapoint[3] = 15 - ai_correct
                    #number of incorrect
                    elif j in (3,4):
                        datapoint[3] = int(match.group(1))
                    #percentage
                    elif j in (5,6):
                        ai_correct = float(match.group(1))
                        datapoint[3] = 15 - (ai_correct*15)/(100)                  
                else:
                    datapoint[3] = "not found"
            except (IndexError, AttributeError):
                datapoint[3] = "not found"

        with open(f"ai_grading1{j}.csv", "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(datapoint)


<>:23: SyntaxWarning: invalid escape sequence '\d'
<>:23: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_59241/1020126406.py:23: SyntaxWarning: invalid escape sequence '\d'
  "(\d+)\u202f%",


[['111010010110111 ', '15', 'very_pessimistic', 'not found'], ['111010010110111 ', '15', 'pessimistic', 'not found'], ['111010010110111 ', '15', 'neutral', 'not found'], ['111010010110111 ', '15', 'confident', 'not found'], ['111010010110111 ', '15', 'very_confident', 'not found'], ['111010010110111 ', '15', 'very_pessimistic', 'not found'], ['111010010110111 ', '15', 'pessimistic', 'not found'], ['111010010110111 ', '15', 'neutral', 'not found'], ['111010010110111 ', '15', 'confident', 'not found']] 6001


In [62]:
with open("ai_grading16.csv", "r") as f:
    ai_grading_list = list(csv.reader(f))
for i, row in enumerate(ai_grading_list[1:]):
    if row[3] == "not found":
        print(f"i={i}")
        print(repr(output[i]))

i=727
'6.67'
i=944
'13.33'
i=2070
'33.33'


In [63]:
with open(f"ai_grading_final_v2.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["answer_key_id", "true_mistakes", "prompt_type", "ai_estimated_mistakes"])
    for i in ai_grading_list[1:]:
        if i[3] != "not found":
                writer.writerow(i)


In [64]:
with open(f"ai_grading_final_v2.csv","r") as f:
    final1 = list(csv.reader(f))
with open(f"ai_grading_final.csv","r") as f:
    final2 = list(csv.reader(f))
final1.extend(final2[1:])
with open(f"ai_grading_final_v3.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["answer_key_id", "true_mistakes", "prompt_type", "ai_estimated_mistakes"])
    for i in final1[1:]:
        writer.writerow(i)
with open(f"ai_grading_final_v3.csv","r") as f:
    final3 = list(csv.reader(f))
print(final3[1:10], len(final3))

[['111010010110111 ', '15', 'very_pessimistic', '15'], ['111010010110111 ', '15', 'pessimistic', '15'], ['111010010110111 ', '15', 'neutral', '15'], ['111010010110111 ', '15', 'confident', '15'], ['111010010110111 ', '15', 'very_confident', '15'], ['111010010110111 ', '15', 'very_pessimistic', '15'], ['111010010110111 ', '15', 'pessimistic', '15'], ['111010010110111 ', '15', 'neutral', '15'], ['111010010110111 ', '15', 'confident', '15']] 7995
